# EP09 — AI in Self-Driving Cars
**COMPSCI 713 · S1 2024 Q9 · 8 marks**

*Reference: Luo, Yang & Yuille (2021). Self-supervised pillar motion learning for autonomous driving.*

Discuss **one** AI algorithm used in self-driving cars and how it contributes to autonomous driving.

---

## Full Answer — LiDAR-Based 3D Object Detection [8 marks]

### Algorithm: PointPillars / 3D Object Detection CNN

Self-driving cars use LiDAR sensors that produce 3D point clouds — millions of (x,y,z) coordinate points per frame representing the surrounding environment.

**PointPillars** (and related architectures like VoxelNet, CenterPoint) process these point clouds with a CNN-based detection pipeline to identify and localise all objects in the scene.

### How it works
1. **Pillar encoding:** the 3D point cloud is divided into vertical columns (pillars) over a    2D ground plane grid. Points within each pillar are aggregated into a fixed-length feature vector
2. **Pseudo-image creation:** the encoded pillars form a 2D feature map — treated like an image
3. **Backbone CNN:** a standard 2D CNN processes the feature map to extract spatial features
4. **Detection head:** predicts 3D bounding boxes (x, y, z, length, width, height, orientation)    and class (car, pedestrian, cyclist, truck) for each detected object
5. **NMS:** Non-Maximum Suppression removes duplicate detections

### Contribution to Autonomous Driving

| Contribution | Details |
|-------------|--------|
| **Real-time perception** | Processes LiDAR frames at >10 Hz, keeping scene awareness current |
| **3D bounding boxes** | Provides position, size, and orientation of each object in 3D space |
| **Multi-class recognition** | Distinguishes pedestrians, cyclists, vehicles — each needs different avoidance behaviour |
| **Motion input** | Bounding boxes feed into Kalman filter trackers to predict future object positions |
| **Sensor fusion** | LiDAR detections merged with camera detections for richer semantic labels |

Without reliable 3D object detection, the planning module cannot safely decide when to brake, change lane, or yield — detection is the **perception backbone** of every autonomous vehicle stack.

### Self-supervised extension (Luo et al. 2021)
Luo et al. proposed learning pillar motion between consecutive frames using a **self-supervised pretext task** — no manual motion annotations needed. The model learns to predict how each pillar moves between frames, producing motion features that improve downstream detection and tracking accuracy.

---

## Code Demo — NMS and IoU (core algorithmic components of object detection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as patches

def iou(a, b):
    'Intersection over Union for boxes [x1,y1,x2,y2].'
    xi1,yi1 = max(a[0],b[0]), max(a[1],b[1])
    xi2,yi2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua>0 else 0.0

def nms(boxes, scores, thr=0.5):
    'Non-Maximum Suppression: keep highest-scoring non-overlapping boxes.'
    order = np.argsort(scores)[::-1]
    keep  = []
    while len(order):
        i = order[0]; keep.append(i)
        ious = np.array([iou(boxes[i], boxes[j]) for j in order[1:]])
        order = order[1:][ious < thr]
    return keep

# Simulated raw detections before NMS
boxes = np.array([
    [40, 50, 160, 180],   # car A  (highest score)
    [45, 55, 165, 185],   # car A  (duplicate — slightly shifted)
    [50, 48, 162, 175],   # car A  (duplicate)
    [250, 70, 380, 200],  # pedestrian (highest)
    [255, 72, 383, 202],  # pedestrian (duplicate)
])
scores = np.array([0.95, 0.78, 0.65, 0.91, 0.60])
labels = ['car','car','car','person','person']

kept = nms(boxes, scores)
print(f'Before NMS: {len(boxes)} detections')
print(f'After  NMS: {len(kept)} detections  (indices {kept})')
for i in kept:
    print(f'  {labels[i]:6s}  score={scores[i]:.2f}  box={boxes[i]}')

# Visualise
fig, axes = plt.subplots(1,2, figsize=(12,5))
bg = np.ones((280,440,3))*0.12  # dark road background
for ax, sel, title in [
        (axes[0], range(len(boxes)), 'Raw detections (before NMS)'),
        (axes[1], kept,              'Final detections (after NMS)')]:
    ax.imshow(bg)
    for i in sel:
        x1,y1,x2,y2 = boxes[i]
        col = 'cyan' if labels[i]=='car' else 'lime'
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=2, edgecolor=col, facecolor='none'))
        ax.text(x1, y1-3, f"{labels[i]} {scores[i]:.2f}", color=col, fontsize=8)
    ax.set_title(title); ax.axis('off')
plt.suptitle('Object Detection in Self-Driving Cars\nNMS removes duplicate boxes', fontsize=11)
plt.tight_layout(); plt.show()